In [0]:
%run ./00_config

dbutils.fs.rm("abfss://gold@stcryptolakehouse.dfs.core.windows.net/crypto_advanced_metrics", True)

In [0]:
%sql
-- 1. Vacina: Garante que conseguimos ler a tabela base
CREATE SCHEMA IF NOT EXISTS hive_metastore.crypto_gold
LOCATION 'abfss://gold@stcryptolakehouse.dfs.core.windows.net/';

CREATE EXTERNAL TABLE IF NOT EXISTS hive_metastore.crypto_gold.crypto_daily_metrics
USING DELTA LOCATION 'abfss://gold@stcryptolakehouse.dfs.core.windows.net/crypto_daily_metrics';

-- 2. Recria a tabela avançada automaticamente substituindo os dados velhos
CREATE OR REPLACE TABLE hive_metastore.crypto_gold.crypto_advanced_metrics
LOCATION 'abfss://gold@stcryptolakehouse.dfs.core.windows.net/crypto_advanced_metrics'
AS
WITH metricas_base AS (
    SELECT moeda_id, data_referencia, preco_medio_usd, volume_total_dia
    FROM hive_metastore.crypto_gold.crypto_daily_metrics
),
calculos_janela AS (
    SELECT moeda_id, data_referencia, preco_medio_usd, volume_total_dia,
        LAG(preco_medio_usd, 1) OVER (PARTITION BY moeda_id ORDER BY data_referencia) AS preco_dia_anterior,
        ROUND(AVG(preco_medio_usd) OVER (PARTITION BY moeda_id ORDER BY data_referencia ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 2) AS media_movel_7d
    FROM metricas_base
)
SELECT 
    moeda_id, 
    data_referencia, 
    preco_medio_usd, 
    media_movel_7d,
    ROUND(((preco_medio_usd - preco_dia_anterior) / NULLIF(preco_dia_anterior, 0)) * 100, 2) AS variacao_percentual_diaria,
    volume_total_dia
FROM calculos_janela
ORDER BY data_referencia DESC, moeda_id;

In [0]:
%sql
-- Consultando os dados recém-criados
SELECT * FROM hive_metastore.crypto_gold.crypto_advanced_metrics;

In [0]:
%sql
SELECT preco_medio_usd 
FROM hive_metastore.crypto_gold.crypto_advanced_metrics
WHERE moeda_id = 'bitcoin'
ORDER BY data_referencia DESC
LIMIT 1;

In [0]:
%sql
SELECT data_referencia, moeda_id, preco_medio_usd, variacao_percentual_diaria, media_movel_7d
FROM hive_metastore.crypto_gold.crypto_advanced_metrics
ORDER BY data_referencia;

In [0]:
%sql
SELECT preco_medio_usd 
FROM hive_metastore.crypto_gold.crypto_advanced_metrics
WHERE moeda_id = 'bitcoin'
ORDER BY data_referencia DESC
LIMIT 1;

In [0]:
%sql
-- Pegando o preço mais atualizado
SELECT preco_medio_usd 
FROM hive_metastore.crypto_gold.crypto_advanced_metrics
WHERE moeda_id = 'bitcoin'
ORDER BY data_referencia DESC
LIMIT 1;

In [0]:
%sql
-- Histórico para o gráfico
SELECT data_referencia, moeda_id, preco_medio_usd, media_movel_7d
FROM hive_metastore.crypto_gold.crypto_advanced_metrics
ORDER BY data_referencia ASC;

In [0]:
%sql
-- Cria uma View (tabela virtual) focada apenas no dia de hoje com horas e minutos
CREATE OR REPLACE VIEW hive_metastore.crypto_gold.vw_powerbi_realtime AS
SELECT 
    moeda_id,
    data_hora_cotacao AS horario, 
    preco_usd,
    volume_24h
FROM hive_metastore.crypto_silver.crypto_prices_clean
WHERE to_date(data_hora_cotacao) = CURRENT_DATE() -- Traz apenas dados de hoje
ORDER BY horario DESC;